# F4 · LLM-judge baseline (Qwen3-8B, 4-bit) — Feba

Runs `src/eval/baselines/llm_judge.py` on the MultiFC test originals + all English twins and on the AVeriTeC test set, rebuilds `results/F4/audit_E0.csv` with all four baselines, and pushes the results to the branch `feba/F4-baselines`.

**Before you start**
1. Runtime → Change runtime type → **T4 GPU**.
2. Colab Secrets (key icon on the left), with notebook access switched on: `GITHUB_TOKEN` (a GitHub token with repo access), `GIT_EMAIL` (your GitHub commit email), `HF_TOKEN`, `WANDB_KEY`.
3. Run the cells in order. Time on a T4: about 40 minutes. If the runtime disconnects, run everything again: the judge resumes from `results/F4/*.partial`.

In [ ]:
# 1. Clone the repo on Feba's F4 branch and install what this notebook needs
from google.colab import userdata
import os
token = userdata.get('GITHUB_TOKEN')
!git clone -q -b feba/F4-baselines https://{token}@github.com/himaallu/Group-Project.git /content/twincheck
%cd /content/twincheck
!pip install -q -r requirements.txt
!pip install -q bitsandbytes accelerate
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
import wandb
wandb.login(key=userdata.get('WANDB_KEY'))
os.environ['WANDB_API_KEY'] = userdata.get('WANDB_KEY')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Quick check on 10 records (test first, CLAUDE.md §1)
!python -m src.eval.baselines.llm_judge --predict data/processed/en/multifc_test.jsonl --out /content/check/preds_llm_judge.jsonl --limit 10 --no-wandb
!head -n 3 /content/check/preds_llm_judge.jsonl

In [ ]:
# 3. MultiFC test originals + all English twins (3,061 records)
!python -m src.eval.baselines.llm_judge --predict data/processed/en/multifc_test.jsonl data/copies/en/test.jsonl --out results/F4/preds_llm_judge.jsonl

In [ ]:
# 4. AVeriTeC test (429 records)
!python -m src.eval.baselines.llm_judge --predict data/processed/en/averitec_test.jsonl --out results/F4/preds_avt_llm_judge.jsonl

In [ ]:
# 5. Rebuild the audit table with all four baselines
!python -m src.eval.baselines.audit --preds results/F4/preds_*.jsonl --gold data/processed/en/multifc_test.jsonl data/copies/en/test.jsonl data/processed/en/averitec_test.jsonl --out results/F4/audit_E0.csv

In [ ]:
# 6. Push the results (only the F4 result files; the .partial checkpoints stay out)
git_email = userdata.get('GIT_EMAIL')   # the email you use for GitHub commits
!git config user.name "Feba Joshy"
!git config user.email "{git_email}"
!git add results/F4/preds_llm_judge.jsonl results/F4/preds_avt_llm_judge.jsonl results/F4/audit_E0.csv
!git commit -m "[F4] Add LLM-judge (Qwen3-8B) predictions and update audit table"
!git push https://{token}@github.com/himaallu/Group-Project.git feba/F4-baselines